# Pipeline SQL + Python — Notebook Exploratório (Curado)

Este notebook tem **caráter exploratório** e foi mantido propositalmente **enxuto** para:
- compreender o dataset;
- validar as regras de negócio do Problema P1;
- registrar checagens básicas de qualidade de dados.

✅ A solução reprodutível/automatizável (pipeline) está em `src/` e `sql/`.


## 1) Setup
Importações mínimas e configuração para análise.

In [ ]:
# Imports
import os
import pandas as pd
import sqlite3


## 2) Leitura do dataset
Carregamento do CSV e inspeção inicial (shape, tipos e amostra).

In [ ]:
# Carrega o dataset
df = pd.read_csv('dataset.csv')


In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 3) Validação das regras de negócio (Problema P1)
Regras:
- Filtrar pacientes com **idade > 50**
- Criar coluna **Perfil**: `Normal` se `BMI < 30`, caso contrário `Obeso`


In [ ]:
df_50 = df[df['Age'] > 50].copy()
df_50.shape


In [ ]:
df_50['Perfil'] = np.where(df_50['BMI'] >= 30, 'Obeso', 'Normal')
df_50[['Age','BMI','Perfil']].head()


In [ ]:
df_50['Perfil'].value_counts(dropna=False)


### Relação rápida com o desfecho (Outcome)
Tabela cruzada simples para observar diferenças entre perfis.

In [ ]:
if 'Outcome' in df_50.columns:
    pd.crosstab(df_50['Perfil'], df_50['Outcome'], normalize='index').round(3)
else:
    print('Coluna Outcome não encontrada no dataset.')


## 4) Qualidade de dados (checks rápidos)
O dataset PIMA é conhecido por usar **0** como placeholder de ausente em algumas colunas.
Aqui registramos a incidência desses casos (ex.: BMI=0).

In [ ]:
cols_to_check = [c for c in ['BMI','Glucose','BloodPressure','SkinThickness','Insulin'] if c in df.columns]
missing_like = {c: float((df[c] == 0).mean()) for c in cols_to_check}
pd.Series(missing_like).sort_values(ascending=False).to_frame('pct_zeros')


**Impacto:** se `BMI = 0` for tratado como valor real, pacientes podem ser classificados como `Normal` indevidamente.
Em produção, isso pode evoluir para tratar zeros como `NULL`/ausente e aplicar imputação/descarta.

## 5) Preview do output esperado
Abaixo está um preview do dataset filtrado (>50 anos) com a coluna `Perfil`. A geração do CSV final e persistência em banco estão implementadas no pipeline (`src/etl.py` + `sql/`).

In [ ]:
df_50.head(10)


## 6) Conclusões
- O recorte **Age > 50** reduz o universo de pacientes analisados.
- A regra de classificação por **BMI** cria uma segmentação simples e útil (`Normal` vs `Obeso`).
- Existem sinais de **dados ausentes codificados como 0** (ex.: `BMI`), que devem ser tratados em evolução do pipeline.

**Próximo passo:** executar o pipeline reprodutível (`src/etl.py`) para gerar `data/processed/resultado.csv`.